In [ ]:
!pip install folium
!pip install geopy
!pip install pandas

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd

route = '/content/drive/MyDrive/Cities analysis/worldcities.csv'

data = pd.read_csv(route)

df_cities = pd.DataFrame(data)

In [13]:
import folium
from geopy.distance import great_circle

def create_map(selected_city, range_km, min_population):
    # Find the selected city in the dataframe
    city_info = df_cities[df_cities['city'] == selected_city].iloc[0]
    origin_location = (city_info['lat'], city_info['lng'])

    # Initialize the total population variable
    total_population = 0

    # Create a folium map centered on the selected city
    map = folium.Map(location=origin_location, zoom_start=5)

    # Draw a circle around the selected city to represent the range
    folium.Circle(origin_location, radius=range_km*1000, color='blue', fill=True, fill_opacity=0.1).add_to(map)

    # Iterate over cities to calculate total population and add markers
    for _, row in df_cities.iterrows():
        city_location = (row['lat'], row['lng'])
        if great_circle(city_location, origin_location).km <= range_km:
            total_population += row['population']
            if row['population'] >= min_population:
              folium.CircleMarker(location=city_location, radius=row['population'] / 1000000 * 5,
                                  color='green', fill=True, fill_color='green',
                                  popup=f"{row['city']}, Pop: {row['population']}").add_to(map)

    # Add a marker for the selected city with a popup including the total population of cities shown
    folium.CircleMarker(location=origin_location, radius=10,  # Fixed size for the origin city marker
                        color='darkblue', fill=True, fill_color='darkblue',
                        popup=(f"Origin: {city_info['city']}, Pop: {city_info['population']}\n"
                               f"Total Pop. in Range: {total_population}")).add_to(map)

    return map

# Example usage: Select Prague, a 500 km range, and a minimum population of 0.5 million
map = create_map('Boston', 500, 100000)
map

In [7]:
import folium
import pandas as pd
from geopy.distance import great_circle

# Load the dataset
route = '/content/drive/MyDrive/Cities analysis/worldcities.csv'
data = pd.read_csv(route)

# Simplified list of European countries ISO2 codes, extend this list as needed
european_countries_iso2 = ['AL', 'AD', 'AM', 'AT', 'BY', 'BE', 'BA', 'BG', 'HR', 'CY', 'CZ', 'DK', 'EE', 'FI', 'FR', 'GE', 'DE', 'GR', 'HU', 'IS', 'IE', 'IT', 'XK', 'LV', 'LI', 'LT', 'LU', 'MT', 'MD', 'MC', 'ME', 'NL', 'MK', 'NO', 'PL', 'PT', 'RO', 'RU', 'SM', 'RS', 'SK', 'SI', 'ES', 'SE', 'CH', 'TR', 'UA', 'GB', 'VA']

# Filter cities in European countries
df_cities = data[data['iso2'].isin(european_countries_iso2)]

# Sort by population and reset index
df_cities = df_cities.sort_values(by='population', ascending=False).reset_index(drop=True)

def create_map_for_top_n_european_cities(n, range_km, min_population):
    # Create a base map
    europe_map = folium.Map(location=[50.1109, 10.4618], zoom_start=4)  # Centered roughly on Europe

    # Filter top n cities
    top_n_cities = df_cities.head(n)

    for _, city_info in top_n_cities.iterrows():
        origin_location = (city_info['lat'], city_info['lng'])
        total_population = 0

        # Draw a circle around each city to represent the range
        folium.Circle(origin_location, radius=range_km*1000, color='blue', fill=True, fill_opacity=0.1).add_to(europe_map)

        # Add a marker for each top city
        folium.CircleMarker(location=origin_location, radius=10,
                            color='darkblue', fill=True, fill_color='darkblue',
                            popup=(f"{city_info['city']}, Pop: {city_info['population']}")).add_to(europe_map)

        # Iterate over cities to calculate total population within range and above minimum population
        for _, row in df_cities.iterrows():
            city_location = (row['lat'], row['lng'])
            if great_circle(city_location, origin_location).km <= range_km:
                total_population += row['population']
                if row['population'] >= min_population:
                    folium.CircleMarker(location=city_location, radius=row['population'] / 1000000 * 5,
                                        color='green', fill=True, fill_color='green',
                                        popup=f"{row['city']}, Pop: {row['population']}").add_to(europe_map)

        # Update the popup for the top city marker to include total population in range
        folium.Marker(location=origin_location,
                      popup=(f"{city_info['city']}, Total Pop. in Range: {total_population}")).add_to(europe_map)

    return europe_map

# Example usage: Create a map for the top 10 European cities with a 500 km range and a minimum population of 500,000
map = create_map_for_top_n_european_cities(30, 100, 500000)
map

In [ ]:
import folium
import pandas as pd
from geopy.distance import great_circle

# Load the dataset
#route = '/content/drive/MyDrive/Vortex Deflector/08_evtol conceptual design/01_Aircraft_Design/02_Market_Study/01_Range_Analysis/worldcities.csv'
data = pd.read_csv(route)

# List of Middle Eastern countries ISO2 codes, including Turkey
middle_east_and_turkey_iso2 = ['TR', 'SA', 'IR', 'IQ', 'AE', 'SY', 'OM', 'IL', 'JO', 'YE', 'LB', 'PS', 'KW', 'QA', 'BH']

# Filter cities in Middle Eastern countries and Turkey
df_cities = data[data['iso2'].isin(middle_east_and_turkey_iso2)]

# Sort by population and reset index
df_cities = df_cities.sort_values(by='population', ascending=False).reset_index(drop=True)

def create_map_for_middle_east_and_turkey(n, range_km, min_population):
    # Create a base map, centered on a point roughly in the middle of the Middle East
    middle_east_map = folium.Map(location=[33.0, 43.0], zoom_start=5)  # Coordinates roughly central to the Middle East

    # Filter top n cities
    top_n_cities = df_cities.head(n)

    for _, city_info in top_n_cities.iterrows():
        origin_location = (city_info['lat'], city_info['lng'])
        total_population = 0

        # Draw a circle around each city to represent the range
        folium.Circle(origin_location, radius=range_km*1000, color='blue', fill=True, fill_opacity=0.1).add_to(middle_east_map)

        # Add a marker for each top city
        folium.CircleMarker(location=origin_location, radius=10,
                            color='darkblue', fill=True, fill_color='darkblue',
                            popup=(f"{city_info['city']}, Pop: {city_info['population']}")).add_to(middle_east_map)

        # Iterate over cities to calculate total population within range and above minimum population
        for _, row in df_cities.iterrows():
            city_location = (row['lat'], row['lng'])
            if great_circle(city_location, origin_location).km <= range_km:
                total_population += row['population']
                if row['population'] >= min_population:
                    folium.CircleMarker(location=city_location, radius=row['population'] / 1000000 * 5,
                                        color='green', fill=True, fill_color='green',
                                        popup=f"{row['city']}, Pop: {row['population']}").add_to(middle_east_map)

        # Update the popup for the top city marker to include total population in range
        folium.Marker(location=origin_location,
                      popup=(f"{city_info['city']}, Total Pop. in Range: {total_population}")).add_to(middle_east_map)

    return middle_east_map

# Example usage: Create a map for the top 10 cities in the Middle East and Turkey with a 500 km range and a minimum population of 500,000
map = create_map_for_middle_east_and_turkey(40, 800, 50000)
map

In [3]:
import folium
from folium import plugins
import pandas as pd
from geopy.distance import great_circle

# Assuming you've already loaded the global dataset of cities into `data`
# data = pd.read_csv('/path/to/your/worldcities.csv')

def find_top_cities_and_inner_ranges(lat, lng, range_km, n, min_population, inner_range_km):
    global data
    # Filter out cities with population below the minimum threshold
    df_cities = data[data['population'] >= min_population].copy()

    # Calculate the distance of each city from the specified location
    df_cities['distance'] = df_cities.apply(lambda row: great_circle((lat, lng), (row['lat'], row['lng'])).km, axis=1)

    # Filter cities within the specified outer range
    cities_in_range = df_cities[df_cities['distance'] <= range_km]

    # Sort by population and select the top n cities
    top_n_cities = cities_in_range.sort_values(by='population', ascending=False).head(n)

    # Create a base map centered on the specified location
    base_map = folium.Map(location=[lat, lng], zoom_start=5)

    # Create a marker cluster
    marker_cluster = plugins.MarkerCluster().add_to(base_map)

    # Add a marker for the selected location with its own inner range
    folium.Marker([lat, lng], popup='Selected Location', icon=folium.Icon(color='red')).add_to(marker_cluster)
    folium.Circle([lat, lng], radius=inner_range_km*1000, color='red', fill=True, fill_opacity=0.2).add_to(base_map)

    for _, city_info in top_n_cities.iterrows():
        city_location = (city_info['lat'], city_info['lng'])

        # Add a marker for each top city with a popup showing its name and population
        popup_text = f"{city_info['city']}, Pop: {int(city_info['population'])}"
        folium.Marker(city_location, popup=popup_text, icon=folium.Icon(color='blue')).add_to(marker_cluster)

        # Draw a circle of influence around each top city representing the inner range
        folium.Circle(city_location, radius=inner_range_km*1000, color='blue', fill=True, fill_opacity=0.2).add_to(base_map)

    return base_map

# Example usage
# Selecting a location, finding the top 5 cities within a 1000 km range,
# considering cities with more than 1 million people, and showing a 200 km inner range around each of those cities.
# 42.58890032918924, -72.31121652712066
map = find_top_cities_and_inner_ranges(42.588, -72.311, 800, 20, 100000, 100)
map

NameError: name 'data' is not defined